# MPI Point-to-Point Communication — Blocking and Nonblocking

The [previous notebook](mpi-in-a-nutshell.ipynb) introduced `MPI_Send()` and `MPI_Recv()` and ended with a deadlock that we could only avoid by carefully ordering the calls.
This notebook looks closer at point-to-point communication: what exactly happens when a message is sent, which guarantees MPI gives, and which tools it offers to write correct and efficient communication patterns.

**After working through this notebook you will be able to**

* distinguish the send modes (standard, buffered, synchronous, ready) and explain when each of them completes,
* explain the ordering and fairness guarantees of MPI,
* recognize *unsafe* programs that rely on message buffering, and test for them,
* implement neighbor exchanges — ring shifts, open chains, ghost cell exchange — with `MPI_Sendrecv()`, `MPI_Sendrecv_replace()` and `MPI_PROC_NULL`,
* receive messages of unknown size with `MPI_Probe()`,
* parallelize a dense matrix-vector multiplication with a ring shift,
* use nonblocking communication (`MPI_Isend()`, `MPI_Irecv()`, `MPI_Wait*()`, `MPI_Test*()`) to avoid deadlocks and to overlap communication with computation.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The C code cells are compiled and run with the `%%mpi` cell magic, which was introduced in [MPI in a Nutshell](mpi-in-a-nutshell.ipynb).
Remember that a cell is the **body of `main()`**, with `rank` and `size` already set up, unless you pass `-r`.
The options used in this notebook are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-T S` | kill the program after `S` seconds (default: 60) — we will produce a few deadlocks |
| `-t` | add timing code (printed by rank 0) |
| `-f FLAG …` | pass additional flags to the compiler, without the leading `-` (e.g. `-f DDEBUG` for `-DDEBUG`) |
| `-o FILE` | where to write the source code |

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

## Recap: Message Matching

![A message travels from the sender, rank i, to the receiver, rank j. It consists of the data (buffer, count, datatype) and an envelope (source, destination, tag, communicator)](img/message.svg)

A receive gets a message only if their **envelopes match**:

| Envelope | Sender specifies | Receiver specifies |
|---|---|---|
| source | implicitly: its own rank | `source` or `MPI_ANY_SOURCE` |
| destination | `dest` | implicitly: its own rank |
| tag | `tag` | `tag` or `MPI_ANY_TAG` |
| communicator | `comm` | `comm` — must be the same, there is no wildcard |

In addition, the **types must match**: for basic data types, sender and receiver use the same type; for derived data types (later in the course), the *sequence* of basic types must match.

Two properties of point-to-point communication in MPI are worth keeping in mind:

* **Push model:** the sender initiates the communication.
  A process cannot fetch data from another one — the owner of the data has to send it.
* **Blocking:** `MPI_Send()` returns once the envelope and the message data have been "stored away", i.e., when the send buffer may be reused.
  As we have seen, this does *not* mean that the message has arrived.
  `MPI_Recv()` returns once the message is in the receive buffer.

## Communication Modes

When a send is *posted*, MPI knows the envelope and the message data; when a receive is posted, MPI knows the envelope of the message it is waiting for.
What happens in between depends on the **communication mode** of the send:

![Timelines of the three send modes. Buffered: the send returns after copying the data, before the receive is posted. Synchronous: the send waits until the receiver has posted the matching receive. Ready: the receive is already posted when the send starts](img/send-modes.svg)

| Mode | Can start without a matching receive? | Completes … | Local? |
|---|---|---|---|
| **buffered** | yes | when the data has been copied — possibly before the matching receive is posted | ✅ local |
| **synchronous** | yes | only once the matching receive has been posted *and* started | ❌ non-local |
| **ready** | **no** — the receive must already be posted | like a standard send — but the receiver is known to be waiting | ❌ non-local |

An operation is **local** if its completion depends only on the calling process; it is **non-local** if it may have to wait for another process to do something.
A non-local operation is what can turn a missing or misplaced call on one process into a deadlock of the whole program.
Apart from its precondition, a ready send behaves like a standard send — which may have to wait for the receiver, as we'll see below.

The **standard mode** of `MPI_Send()` does not appear in the table: it is *either* buffered *or* synchronous — it's up to the MPI implementation.
We'll come back to this below.

## Communication Semantics

![Order: two sends from rank 0 to rank 1 are received in the order they were sent. Fairness: rank 3 receives from any source and the message from rank 0 may never be received, because messages from ranks 1 and 2 are always matched first](img/semantics.svg)

MPI guarantees that messages are **non-overtaking**:
if a sender sends two messages to the same destination and both match the same receive, the receive gets the first one.
This only holds for one pair of processes and one communicator — messages from *different* senders are not ordered.

MPI does **not** guarantee **fairness**:
if several senders compete for the same receive (e.g., with `MPI_ANY_SOURCE`), one of them may never be served while the others keep sending.
Avoiding such starvation is the programmer's job.

> 🤔 **Predict:** Rank 0 sends the numbers `0, 1, 2, 3, 4` to rank 1, one message each, all with the same tag. In which order does rank 1 receive them?

In [ ]:
%%mpi -n 2 -o code/p2p-non-overtaking.c
if (rank == 0) {
    for (int i = 0; i < 5; ++i)
        MPI_Send(&i, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
} else if (rank == 1) {
    for (int i = 0; i < 5; ++i) {
        int value;
        MPI_Recv(&value, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("rank 1: message %d contains %d\n", i, value);
    }
}

The order is guaranteed only among messages that **match the same receive**.
With different tags, the receiver is free to pick the messages in any order.

> 🤔 **Predict:** Rank 0 sends a message with tag 1, then one with tag 2. Rank 1 first receives tag 2, then tag 1. Does this work, or does it deadlock?

In [ ]:
%%mpi -n 2 -T 5 -o code/p2p-tag-order.c
if (rank == 0) {
    int a = 1, b = 2;
    MPI_Send(&a, 1, MPI_INT, 1, 1, MPI_COMM_WORLD);     // tag 1 first ...
    MPI_Send(&b, 1, MPI_INT, 1, 2, MPI_COMM_WORLD);     // ... then tag 2
} else if (rank == 1) {
    int a, b;
    MPI_Recv(&b, 1, MPI_INT, 0, 2, MPI_COMM_WORLD, MPI_STATUS_IGNORE);   // tag 2 first!
    MPI_Recv(&a, 1, MPI_INT, 0, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    printf("rank 1: a = %d (tag 1), b = %d (tag 2)\n", a, b);
}

It most likely works — but only because the first `MPI_Send()` returns before its message is received: the small message is buffered by MPI.
Remember this example; we'll test it again in the section on *safe* programs.

Now let three ranks compete for the receives of rank 0.
Each sender sends 1000 messages; rank 0 receives with `MPI_ANY_SOURCE` and counts where the first 1000 messages came from.

> 🤔 **Predict:** Will every sender get about a third of the first 1000 receives? Run the cell a few times.

In [ ]:
%%mpi -n 4 -o code/p2p-fairness.c
const int msgs = 1000;      // messages per sender

if (rank != 0) {
    for (int i = 0; i < msgs; ++i)
        MPI_Send(&rank, 1, MPI_INT, 0, 0, MPI_COMM_WORLD);
} else {
    int *count = calloc(size, sizeof(int));
    int source;

    // the first msgs messages: whoever comes first ...
    for (int i = 0; i < msgs; ++i) {
        MPI_Recv(&source, 1, MPI_INT, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        ++count[source];
    }
    for (int r = 1; r < size; ++r)
        printf("%4d of the first %d messages came from rank %d\n", count[r], msgs, r);

    // ... and all remaining ones, so that every sender can finish
    for (int i = msgs; i < (size - 1) * msgs; ++i)
        MPI_Recv(&source, 1, MPI_INT, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    free(count);
}

The distribution depends on the timing of the processes and on the internals of the MPI library — it is often far from even.
If every sender has to be served in turn, receive from specific sources (as in the ordered output of the previous notebook), or post one nonblocking receive per sender and complete them with `MPI_Waitsome()`, which is shown later in this notebook.

## Blocking Send Variants

For every communication mode, MPI provides a separate send call.
They all take exactly the same arguments as `MPI_Send()`:

| Call | Mode | Completes when … |
|---|---|---|
| `MPI_Send()` | standard | depends on the implementation: buffered or synchronous |
| `MPI_Bsend()` | buffered | the data has been copied into the attached buffer |
| `MPI_Ssend()` | synchronous | the matching receive has started |
| `MPI_Rsend()` | ready | the data has been sent — the matching receive **must** already be posted |
| `MPI_Recv()` | — | the message has been received |

All of these are **blocking**: once the call returns, the buffer can safely be reused.
There is only one receive call — it matches any of the send modes.

The **standard send** `MPI_Send()` is buffered for small messages and synchronous for large ones (*eager* and *rendezvous* protocol in the previous notebook).
Where the limit lies depends on the implementation, the transport and other factors — that's why it is not specified.

`MPI_Rsend()` exists for implementations that can save a handshake when they know that the receive is already posted.
In practice this hardly gives any benefit, and a ready send without a posted receive is an error MPI usually doesn't detect — avoid it.

### Synchronous Send

`MPI_Ssend()` performs a **handshake** with the receiver: it only returns once the matching receive has started.
This synchronizes the two processes — which costs time and, as we'll see, exposes deadlocks.

Let's return to the exchange between two processes from the previous notebook: both first send, then receive.
With `MPI_Send()` and a single `int`, this worked, because the message was buffered.

> 🤔 **Predict:** What happens with `MPI_Ssend()`?

In [ ]:
%%mpi -n 2 -T 5 -o code/p2p-ssend-deadlock.c
int other = 1 - rank;       // the partner process (run with exactly 2 processes)
int sendval = rank, recvval;

MPI_Ssend(&sendval, 1, MPI_INT, other, 0, MPI_COMM_WORLD);
MPI_Recv(&recvval, 1, MPI_INT, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

printf("rank %d: received %d\n", rank, recvval);

Both processes wait in `MPI_Ssend()` for the other one to start its receive — a deadlock, independent of the message size.

### Buffered Send

`MPI_Bsend()` completes as soon as the message has been copied to a buffer.
This buffer is **provided by the user** and has to be attached to MPI before the first buffered send:

```c
int MPI_Buffer_attach(void *buffer, int size);
int MPI_Buffer_detach(void *buffer_addr, int *size);
```

* The buffer must hold all messages that are pending at the same time, plus `MPI_BSEND_OVERHEAD` bytes of bookkeeping per message.
  `MPI_Pack_size()` tells you how many bytes a message needs.
* Only one buffer can be attached per process (MPI 4.1 also allows buffers per communicator or session — not yet available in all libraries).
* `MPI_Buffer_detach()` blocks until all messages in the buffer have been delivered.
  Its first argument is the *address* of the pointer, since it returns the buffer's address.

With buffered sends, the head-to-head exchange works even for large messages:

In [ ]:
%%mpi -n 2 -T 10 -o code/p2p-bsend.c
int other = 1 - rank;       // the partner process (run with exactly 2 processes)
const int n = 1 << 20;      // 1 Mi doubles = 8 MiB, far above the eager limit

double *sendbuf = calloc(n, sizeof(double));
double *recvbuf = malloc(n * sizeof(double));

// attach a buffer that is large enough for one message
int bufsize;
MPI_Pack_size(n, MPI_DOUBLE, MPI_COMM_WORLD, &bufsize);
bufsize += MPI_BSEND_OVERHEAD;
void *buffer = malloc(bufsize);
MPI_Buffer_attach(buffer, bufsize);

MPI_Bsend(sendbuf, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD);
MPI_Recv(recvbuf, n, MPI_DOUBLE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
printf("rank %d: exchanged %zu bytes\n", rank, n * sizeof(double));

// waits until the buffered message has been delivered
MPI_Buffer_detach(&buffer, &bufsize);
free(buffer);
free(sendbuf);
free(recvbuf);

> 🤔 **Predict:** What happens if you remove the line `bufsize += MPI_BSEND_OVERHEAD;`? Try it.

Buffered sends are rarely the best solution:
every message is copied once more, the buffer has to be managed by hand, and if it overflows, `MPI_Bsend()` fails with an error — which, by default, aborts the program.

## Safe Programs

A frequent pattern in message passing is the **ring shift**: every process sends data to its right neighbor and receives data from its left neighbor, with periodic boundaries.

![Ring shift with four ranks: every rank sends its send buffer to the receive buffer of its right neighbor, and rank 3 sends to rank 0](img/ring-shift.svg)

The neighbors can be computed with the modulo operator:

```c
int left  = (rank - 1 + size) % size;   // my left neighbor
int right = (rank + 1) % size;          // my right neighbor
```

The obvious implementation lets every process send first and receive afterwards.

> 🤔 **Predict:** Up to which message size does it work?

In [ ]:
%%mpi -n 4 -T 10 -o code/p2p-ring-send-recv.c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

for (int n = 1; n <= (1 << 20); n *= 16) {
    int *sendbuf = malloc(n * sizeof(int));
    int *recvbuf = malloc(n * sizeof(int));
    for (int i = 0; i < n; ++i)
        sendbuf[i] = rank;

    MPI_Send(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD);
    MPI_Recv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    if (rank == 0)
        printf("%9zu bytes: rank 0 received data from rank %d\n", n * sizeof(int), recvbuf[0]);

    free(sendbuf);
    free(recvbuf);
}

As in the previous notebook, the program deadlocks as soon as `MPI_Send()` switches from buffered to synchronous mode — now with all processes waiting for their right neighbor, in a cycle.

A program is called **safe** if it does not rely on message buffering to complete.
Equivalently: a safe program still runs correctly if every `MPI_Send()` is replaced by `MPI_Ssend()`.
**Best practice: use the standard send, but make sure your program is safe.**

This gives a simple test.
Define a macro that replaces `MPI_Send` by `MPI_Ssend` — here with the compiler flag `-DMPI_Send=MPI_Ssend`, passed via `-f`:

In [ ]:
%%mpi -n 4 -T 5 -f DMPI_Send=MPI_Ssend
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;
int sendval = rank, recvval;

MPI_Send(&sendval, 1, MPI_INT, right, 0, MPI_COMM_WORLD);       // this is MPI_Ssend now
MPI_Recv(&recvval, 1, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

printf("rank %d: received %d\n", rank, recvval);

Already a single `int` deadlocks — the test reveals the bug without having to find the eager limit.

> 🤔 **Predict:** Is the example with the two tags above safe? Add `-f DMPI_Send=MPI_Ssend` to its cell and check.

One way to make the exchange safe is to order the calls so that every send meets a posted receive — e.g., for two processes, one of them sends first and the other one receives first.
For the ring, see Exercise 2.
But there is a much simpler way.

## Combining Send and Receive: `MPI_Sendrecv()`

```c
int MPI_Sendrecv(const void *sendbuf, int sendcount, MPI_Datatype sendtype, int dest,   int sendtag,
                 void       *recvbuf, int recvcount, MPI_Datatype recvtype, int source, int recvtag,
                 MPI_Comm comm, MPI_Status *status);
```

The arguments are simply those of `MPI_Send()` followed by those of `MPI_Recv()`.
**MPI takes care that no deadlock occurs** — it performs the send and the receive as if they ran concurrently.

* Send and receive buffers must be **disjoint**; they may have different counts and data types.
* `MPI_Sendrecv()` matches any send and receive call on the other side: the partner may call `MPI_Sendrecv()`, but also `MPI_Recv()` and `MPI_Send()`, or the nonblocking variants.
* The source and the destination may be different processes — as in the ring shift.

The ring shift becomes a single call that works for any message size:

In [ ]:
%%mpi -n 4 -o code/p2p-ring-sendrecv.c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

const int n = 1 << 20;      // 4 MiB, far above the eager limit
int *sendbuf = malloc(n * sizeof(int));
int *recvbuf = malloc(n * sizeof(int));
for (int i = 0; i < n; ++i)
    sendbuf[i] = rank;

MPI_Sendrecv(sendbuf, n, MPI_INT, right, 0,
             recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

printf("rank %d: received data from rank %d\n", rank, recvbuf[0]);

free(sendbuf);
free(recvbuf);

### Open Chains: `MPI_PROC_NULL`

Without periodic boundaries, the first process has no left and the last process has no right neighbor.
Instead of special cases in the code, use the *null process* `MPI_PROC_NULL` as the neighbor:

![Open chain: rank 0 receives from MPI_PROC_NULL, so its receive buffer stays unchanged; rank 3 sends to MPI_PROC_NULL, so nothing is sent](img/open-chain.svg)

* A send to or a receive from `MPI_PROC_NULL` is a **no-op**: it returns as soon as possible and does not alter the buffer.
* A receive from `MPI_PROC_NULL` sets the status to source `MPI_PROC_NULL`, tag `MPI_ANY_TAG` and count 0.
* This works for all point-to-point calls, not only for `MPI_Sendrecv()`.

> 🤔 **Predict:** What does each rank print?

In [ ]:
%%mpi -n 4 -o code/p2p-open-chain.c
int left  = rank - 1;    if (left < 0)      left  = MPI_PROC_NULL;
int right = rank + 1;    if (right >= size) right = MPI_PROC_NULL;

int sendval = 10 * rank, recvval = -1;
MPI_Status status;

MPI_Sendrecv(&sendval, 1, MPI_INT, right, 0,
             &recvval, 1, MPI_INT, left,  0, MPI_COMM_WORLD, &status);

int count;
MPI_Get_count(&status, MPI_INT, &count);
printf("rank %d: recvval = %2d, count = %d%s\n", rank, recvval, count,
       status.MPI_SOURCE == MPI_PROC_NULL ? " (received from MPI_PROC_NULL)" : "");

Rank 0 receives from `MPI_PROC_NULL`: its `recvval` keeps the initial value −1, the count is 0, and the status reports `MPI_PROC_NULL` as the source.
Rank 3 sends to `MPI_PROC_NULL`, so its value 30 goes nowhere.
All other ranks receive ten times the rank of their left neighbor.

## Pattern: Ghost Cell Exchange

Many iterative algorithms — stencil codes, finite differences, image filters — update every cell of a grid from the values of its neighbors.
To parallelize them, the grid is split into **tiles** and each process works on one of them (*domain decomposition*).
The cells at the edge of a tile need values that belong to the neighboring tiles.
Therefore every tile is surrounded by a layer of **ghost cells** (also *halo*), holding copies of the neighbors' boundary cells.

![Left: a 2D domain distributed to 4 by 3 ranks. Middle: a tile surrounded by ghost cells. Right: two neighboring tiles i and j each send their boundary layer to the ghost cells of the other](img/ghost-cells.svg)

After each sweep over its tile, a process performs a **ghost cell exchange** with each of its neighbors:

1. copy the new values of the boundary layer into a contiguous send buffer,
2. send it to the neighbor and receive the neighbor's boundary layer from the same neighbor — a perfect job for `MPI_Sendrecv()`,
3. copy the received data into the ghost cells.

Here is the 1D version: every rank owns `N` cells `u[1] … u[N]` with one ghost cell on each side.
Since only single values are exchanged, the copy steps 1 and 3 are not needed — MPI sends directly from and receives directly into the array.
At the ends of the global domain, `MPI_PROC_NULL` leaves the ghost cells unchanged.

In [ ]:
%%mpi -n 4 -o code/p2p-ghost-cells-1d.c
#define N 4     // own cells per rank

double u[N + 2];                            // u[0] and u[N + 1] are ghost cells
u[0] = u[N + 1] = -1.0;                     // marks ghost cells that were not updated
for (int i = 1; i <= N; ++i)
    u[i] = rank * N + i - 1;                // the global index of the cell

int left  = rank > 0        ? rank - 1 : MPI_PROC_NULL;
int right = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;

// send my last own cell to the right, receive my left ghost cell from the left ...
MPI_Sendrecv(&u[N], 1, MPI_DOUBLE, right, 0,
             &u[0], 1, MPI_DOUBLE, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
// ... and the other way round
MPI_Sendrecv(&u[1],     1, MPI_DOUBLE, left,  1,
             &u[N + 1], 1, MPI_DOUBLE, right, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

// prints exactly N = 4 own cells
printf("rank %d: %5.1f | %4.1f %4.1f %4.1f %4.1f | %5.1f\n",
       rank, u[0], u[1], u[2], u[3], u[4], u[5]);

In 2D, the boundary layers on the left and right side of a tile are *columns*, which are not contiguous in memory (in C, arrays are stored row by row).
That's where the copy steps come in — or *derived data types*, which let MPI do the copying, and which you'll learn about later in the course.

## In-Place Exchange: `MPI_Sendrecv_replace()`

If the data that is sent is no longer needed afterwards, a single buffer is sufficient:

```c
int MPI_Sendrecv_replace(void *buf, int count, MPI_Datatype datatype,
                         int dest, int sendtag, int source, int recvtag,
                         MPI_Comm comm, MPI_Status *status);
```

![Ring shift with MPI_Sendrecv_replace: each rank sends its buffer to the right neighbor and receives the buffer of its left neighbor into the same buffer](img/sendrecv-replace.svg)

* The same buffer, count and data type are used for sending and receiving.
* Again, MPI makes sure that no deadlock occurs.
* It saves you a buffer — not necessarily time: internally, MPI may need a temporary buffer and an additional copy.

In [ ]:
%%mpi -n 4 -o code/p2p-sendrecv-replace.c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

int buf[3] = { rank, 10 * rank, 100 * rank };

MPI_Sendrecv_replace(buf, 3, MPI_INT, right, 0, left, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

printf("rank %d: buf = { %d, %d, %d }\n", rank, buf[0], buf[1], buf[2]);

## Probing for Messages *(optional)*

Sometimes the receiver does not know how large an incoming message will be.
A *probe* checks for a matching message **without receiving it**:

```c
int MPI_Probe(int source, int tag, MPI_Comm comm, MPI_Status *status);
int MPI_Iprobe(int source, int tag, MPI_Comm comm, int *flag, MPI_Status *status);
```

* `MPI_Probe()` is non-local: it returns only once a matching message is available.
* `MPI_Iprobe()` returns immediately and sets `flag` to true if a matching message is available.
* In both cases, the status contains source, tag and — via `MPI_Get_count()` — the size of the message.

This makes it possible to allocate a buffer of exactly the right size before receiving the message.
Note that the receive must use the source and tag from the status: with wildcards, it could match a different message than the one that was probed.

In [ ]:
%%mpi -n 4 -o code/p2p-probe.c
if (rank != 0) {
    // every rank sends a message of a different length
    int n = 10 * rank;
    double *data = malloc(n * sizeof(double));
    for (int i = 0; i < n; ++i)
        data[i] = rank;
    MPI_Send(data, n, MPI_DOUBLE, 0, 0, MPI_COMM_WORLD);
    free(data);
} else {
    for (int i = 1; i < size; ++i) {
        MPI_Status status;
        int n;

        // wait for any message, but don't receive it yet ...
        MPI_Probe(MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, &status);
        MPI_Get_count(&status, MPI_DOUBLE, &n);

        // ... allocate a buffer of the right size and receive exactly the probed message
        double *data = malloc(n * sizeof(double));
        MPI_Recv(data, n, MPI_DOUBLE, status.MPI_SOURCE, status.MPI_TAG, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("received %2d doubles from rank %d\n", n, status.MPI_SOURCE);
        free(data);
    }
}

If several threads of a process receive messages, another thread could receive the probed message in between.
The *matched* probe `MPI_Mprobe()` together with `MPI_Mrecv()` avoids this; see the MPI standard (section 3.8) for details.

The standard also defines `MPI_Cancel()` to cancel a pending nonblocking communication.
It is rarely needed; cancelling a *send* is deprecated since MPI 4.0.

## Case Study: Parallel Dense Matrix-Vector Multiplication

The dense matrix-vector multiplication (MVM) $y = y + A x$ with an $N \times N$ matrix $A$ computes
$$y_r = y_r + \sum_{c=0}^{N-1} A_{rc}\, x_c \qquad \text{for } r = 0, \dots, N-1.$$
With `a` stored row by row, the serial code is

```c
for (int r = 0; r < N; ++r)
    for (int c = 0; c < N; ++c)
        y[r] += a[r * N + c] * x[c];
```

On a shared-memory system, the rows could simply be distributed among threads, which all read the complete vector `x`.
With MPI, however, **all data is distributed**: each rank stores a block of rows of $A$ and the matching chunks of $y$ and $x$.

![y = y + A times x, with the rows of A and the chunks of y and x colored by the rank that owns them: rank 0 owns the first row block, rank 3 the last](img/mvm-distribution.svg)

Rank $r$ can compute its chunk of $y$ only partially with its own chunk of $x$ — the diagonal block $A_{rr}$.
For the other blocks, it needs the chunks of all other ranks.
Instead of collecting the complete vector on every rank, the chunks of $x$ travel around in a **ring shift**:

![Four steps of the parallel MVM. In step 0, each rank multiplies its diagonal block with its own chunk of x. After each step, the chunks of x are shifted up by one rank, so that in step s rank r multiplies block A(r, r+s mod 4) with chunk x(r+s mod 4)](img/mvm-ring.svg)

After `size` steps, every rank has seen every chunk of `x` and its chunk of `y` is complete.
Since `x` is not needed after it has been passed on, `MPI_Sendrecv_replace()` is the natural choice for the ring shift.

When $N$ is not divisible by the number of processes, the first `N % size` ranks get one row more.
All chunks of `x` then fit into a buffer of the size of the largest chunk — and that's the count used for every shift, since `MPI_Sendrecv_replace()` needs the same count for sending and receiving.

In [ ]:
%%mpi -n 4 -o code/p2p-mvm.c
const int N = 2000;     // dimension of the matrix

// distribute the rows as evenly as possible: the first N % size ranks get one row more
const int num = N / size, rest = N % size;
#define CHUNK_SIZE(r)  (num + ((r) < rest ? 1 : 0))
#define CHUNK_START(r) ((r) * num + ((r) < rest ? (r) : rest))

const int nlocal = CHUNK_SIZE(rank);    // number of local rows
const int start  = CHUNK_START(rank);   // global index of the first local row
const int nmax   = CHUNK_SIZE(0);       // the largest chunk: x must be able to hold any chunk

double *a = malloc((size_t)nlocal * N * sizeof(double));    // local rows of A
double *y = calloc(nlocal, sizeof(double));                // local chunk of y
double *x = calloc(nmax, sizeof(double));                  // the chunk of x currently held

// initialize with known values: A_rc = 1 / (r + c + 1), x_c = c + 1
for (int r = 0; r < nlocal; ++r)
    for (int c = 0; c < N; ++c)
        a[(size_t)r * N + c] = 1.0 / (start + r + c + 1);
for (int c = 0; c < nlocal; ++c)
    x[c] = start + c + 1;

int upper = (rank - 1 + size) % size;   // the chunk of x is passed on to this rank ...
int lower = (rank + 1) % size;          // ... and the next one comes from this rank
int owner = rank;                       // the rank the current chunk of x belongs to

// time only the MVM, not the setup and the check
MPI_Barrier(MPI_COMM_WORLD);
double t0 = MPI_Wtime();

for (int step = 0; step < size; ++step) {
    // multiply the matching block of A with the current chunk of x
    const int cs = CHUNK_START(owner), nc = CHUNK_SIZE(owner);
    for (int r = 0; r < nlocal; ++r)
        for (int c = 0; c < nc; ++c)
            y[r] += a[(size_t)r * N + cs + c] * x[c];

    // ring shift of x - not needed after the last step
    if (step < size - 1) {
        MPI_Sendrecv_replace(x, nmax, MPI_DOUBLE, upper, 0, lower, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        owner = (owner + 1) % size;
    }
}

double t1 = MPI_Wtime();

// check the local rows against a serial computation
double maxerr = 0.0;
for (int r = 0; r < nlocal; ++r) {
    double ref = 0.0;
    for (int c = 0; c < N; ++c)
        ref += 1.0 / (start + r + c + 1) * (c + 1);
    double err = (y[r] - ref) / ref;
    if (err < 0.0) err = -err;
    if (err > maxerr) maxerr = err;
}
printf("rank %d: rows [%4d, %4d), max. relative error %.1e, MVM %.3f ms\n",
       rank, start, start + nlocal, maxerr, 1e3 * (t1 - t0));

free(a);
free(y);
free(x);

Questions:
* Run the cell with `-n 1`, `-n 3` and `-n 4`. Is the result correct when `N` is not divisible by the number of processes?
* Every rank sends `size - 1` chunks of about `N / size` elements, i.e., about `N` elements in total, while it performs about `N² / size` multiply-adds.
  How does the ratio of communication to computation change with `N` and with the number of processes?
* During the ring shift, the processes are not computing. How could the communication be overlapped with the computation? (→ Exercise 3, after the section on nonblocking communication.)

## Summary: Blocking Point-to-Point Communication

* A blocking call is **locally complete** when it returns: the send or receive buffer can be reused safely.
* Send modes:
    * **synchronous** (`MPI_Ssend()`): handshake with the receiver — performance drawbacks, deadlock dangers,
    * **buffered** (`MPI_Bsend()`): completes once the data is copied to a user-provided buffer — additional copies, buffer management,
    * **standard** (`MPI_Send()`): buffered or synchronous, depending on the message size and the implementation.
* Messages between a pair of processes are **non-overtaking**, but there is **no fairness**.
* `MPI_Sendrecv()` and `MPI_Sendrecv_replace()` exchange data without deadlocks; `MPI_PROC_NULL` handles open boundaries.

> **Best practice:** use the standard send and make sure your program is **safe**, i.e., it does not rely on message buffering.
> Test it by replacing `MPI_Send()` with `MPI_Ssend()`.

## Nonblocking Point-to-Point Communication

A **nonblocking** call only *starts* a communication and returns immediately — the communication is completed later by a separate call.
In the meantime, the process can do other things:

![Sequence diagram of the best case: rank 0 calls MPI_Isend, gets a request back immediately and does useful work while MPI waits for rank 1 to post its receive. Rank 0 then calls MPI_Wait, which returns once the message has been transferred](img/nonblocking.svg)

This opens up several opportunities:
* **avoiding deadlocks** — no process waits in a send while its partner waits in a send as well,
* **truly bidirectional communication** — sending and receiving at the same time,
* **avoiding idle time and synchronization** — a process does not have to wait for its partner to reach the matching call,
* **overlapping communication with useful work** — if the MPI implementation and the hardware support it.

### Standard Nonblocking Send and Receive

```c
int MPI_Isend(const void *buf, int count, MPI_Datatype datatype, int dest,   int tag,
              MPI_Comm comm, MPI_Request *request);
int MPI_Irecv(void *buf,       int count, MPI_Datatype datatype, int source, int tag,
              MPI_Comm comm, MPI_Request *request);
```

The `I` stands for *immediate*.
The arguments are the same as for the blocking calls, plus a **request** handle:

* `request` points to a variable of type `MPI_Request`, which is associated with the started operation.
  It is used later to check whether the operation has completed.
* **The return of the call does not imply completion.**
  Until the operation has completed, the buffer must not be modified (`MPI_Isend()`) or used at all (`MPI_Irecv()`).
* `MPI_Irecv()` has no status argument — the status is obtained when the operation completes.
* Every send mode has a nonblocking variant, and all of them match any receive:

| Nonblocking call | Blocking call | Mode | Completes when … |
|---|---|---|---|
| `MPI_Isend()` | `MPI_Send()` | standard | depends on the implementation |
| `MPI_Ibsend()` | `MPI_Bsend()` | buffered | the data has been copied to the buffer |
| `MPI_Issend()` | `MPI_Ssend()` | synchronous | the matching receive has started |
| `MPI_Irsend()` | `MPI_Rsend()` | ready | the data has been sent |
| `MPI_Irecv()` | `MPI_Recv()` | — | the message has been received |

Once completed, a nonblocking operation has exactly the same effect as the corresponding blocking call.

> 🤔 **Predict:** Rank 1 starts a receive and prints the receive buffer right away, and again after the receive has completed. Rank 0 sends its message after a short delay. What is printed?

In [ ]:
%%mpi -n 2 -o code/p2p-irecv.c
int value = -1;

if (rank == 0) {
    double t0 = MPI_Wtime();
    while (MPI_Wtime() - t0 < 0.2)
        ;                                       // busy for 0.2 s
    value = 42;
    MPI_Send(&value, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
} else if (rank == 1) {
    MPI_Request request;
    MPI_Irecv(&value, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, &request);

    // erroneous: the buffer must not be accessed before the receive has completed!
    printf("rank 1: after MPI_Irecv(): value = %d\n", value);

    MPI_Wait(&request, MPI_STATUS_IGNORE);
    printf("rank 1: after MPI_Wait():  value = %d\n", value);
}

The first `printf()` reads the buffer while MPI may be writing to it — its result is undefined.
Here it most likely shows −1, since the message is sent only 0.2 s later; had the message already been waiting, it might show 42.
Only after `MPI_Wait()` is the value guaranteed to be 42.

### Testing for Completion

There are two ways to complete a nonblocking operation:

* **wait** — `MPI_Wait*()` blocks until the operation has completed and the buffer can be reused,
* **test** — `MPI_Test*()` returns immediately and reports whether the operation has completed.

Despite the naming, both pertain to *nonblocking* point-to-point communication.
For a single request:

```c
int MPI_Wait(MPI_Request *request, MPI_Status *status);
int MPI_Test(MPI_Request *request, int *flag, MPI_Status *status);
```

| Argument | Meaning |
|---|---|
| `request` | address of the request handle |
| `flag` | set to true if the operation has completed |
| `status` | address of a status object (as for `MPI_Recv()`), or `MPI_STATUS_IGNORE` |

When the operation has completed, the request is freed and set to `MPI_REQUEST_NULL`.
A wait or test on `MPI_REQUEST_NULL` returns immediately.

The two typical usage patterns are

```c
MPI_Request request;
MPI_Isend(sendbuf, count, MPI_CHAR, dst, 0, MPI_COMM_WORLD, &request);

// do some work ... but don't modify sendbuf

MPI_Wait(&request, MPI_STATUS_IGNORE);
// sendbuf may be reused
```

and

```c
MPI_Request request;
int flag;
MPI_Isend(sendbuf, count, MPI_CHAR, dst, 0, MPI_COMM_WORLD, &request);

do {
    // do some work ... but don't modify sendbuf
    MPI_Test(&request, &flag, MPI_STATUS_IGNORE);
} while (!flag);
// sendbuf may be reused
```

Here, rank 0 starts a large send and keeps working until it has completed, while rank 1 posts the receive only after 0.5 s.

> 🤔 **Predict:** When does the send of rank 0 complete?

In [ ]:
%%mpi -n 2 -o code/p2p-test.c
const int n = 1 << 24;      // 16 MiB, far above the eager limit
char *buf = calloc(n, 1);

if (rank == 0) {
    MPI_Request request;
    int flag = 0;
    long iterations = 0;
    double t0 = MPI_Wtime();

    MPI_Isend(buf, n, MPI_BYTE, 1, 0, MPI_COMM_WORLD, &request);
    do {
        ++iterations;                           // "useful work", which must not modify buf
        MPI_Test(&request, &flag, MPI_STATUS_IGNORE);
    } while (!flag);

    printf("rank 0: send completed after %.3f s and %ld iterations\n", MPI_Wtime() - t0, iterations);
} else if (rank == 1) {
    double t0 = MPI_Wtime();
    while (MPI_Wtime() - t0 < 0.5)
        ;                                       // busy for 0.5 s before posting the receive
    MPI_Recv(buf, n, MPI_BYTE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    printf("rank 1: message received after %.3f s\n", MPI_Wtime() - t0);
}

free(buf);

The 16 MiB message is sent with the rendezvous protocol: the send cannot complete before rank 1 has posted its receive, i.e., after about 0.5 s plus the transfer time.
Rank 0 was not blocked in the meantime — and its repeated calls to `MPI_Test()` are what gives MPI the chance to make progress on the transfer (more on that in Exercise 4).

### Completing Multiple Requests

Usually, several operations are pending at the same time.
They can be completed together:

```c
int MPI_Waitall(int count, MPI_Request requests[], MPI_Status statuses[]);
int MPI_Testall(int count, MPI_Request requests[], int *flag, MPI_Status statuses[]);
```

`MPI_Waitall()` waits until all `count` requests in the array have completed; `MPI_Testall()` sets `flag` to true only if all of them have completed.
The statuses are returned in an array of the same length — or pass `MPI_STATUSES_IGNORE` (note the plural).

With nonblocking calls, the ring shift is deadlock-free, too.
It is good practice to **post the receive first**: then MPI knows where the data has to go when the message arrives and can avoid buffering it.

In [ ]:
%%mpi -n 4 -o code/p2p-ring-nonblocking.c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

const int n = 1 << 20;      // 4 MiB, far above the eager limit
int *sendbuf = malloc(n * sizeof(int));
int *recvbuf = malloc(n * sizeof(int));
for (int i = 0; i < n; ++i)
    sendbuf[i] = rank;

MPI_Request requests[2];
MPI_Irecv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, &requests[0]);
MPI_Isend(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD, &requests[1]);

// work that does not touch sendbuf or recvbuf could be done here

MPI_Waitall(2, requests, MPI_STATUSES_IGNORE);
printf("rank %d: received data from rank %d\n", rank, recvbuf[0]);

free(sendbuf);
free(recvbuf);

### Ghost Cell Exchange with Nonblocking Communication

Nonblocking calls let a process exchange ghost cells with **all neighbors at once**, and update the inner cells of its tile in the meantime:

![A tile with its four neighbors. Each of the four boundary layers is sent to the corresponding neighbor with MPI_Isend, and each layer of ghost cells is received from the corresponding neighbor with MPI_Irecv. The bulk of the tile does not need any ghost cells](img/halo-exchange.svg)

1. copy the new boundary values into contiguous send buffers,
2. start nonblocking receives from and sends to all neighbors,
3. update the cells that do not need ghost cells (*bulk update*),
4. wait with `MPI_Waitall()` for all requests to complete,
5. copy the received data into the ghost cells,
6. update the cells that need the ghost cells.

The bulk update in step 3 gives the communication an **opportunity to overlap** with computation — whether it really progresses in the background depends on the MPI implementation.

Here is the 1D version for a simple smoothing step $u^{\text{new}}_i = (u_{i-1} + u_i + u_{i+1}) / 3$.
For a linear function, the average reproduces the original values exactly, so any error in the ghost cells shows up as a deviation.
The ghost cells start with bogus values that must be overwritten by the exchange — except at the global boundary, where they are fixed.

In [ ]:
%%mpi -n 4 -o code/p2p-halo-overlap.c
#define N 6     // own cells per rank

double u[N + 2], unew[N + 2];               // u[0] and u[N + 1] are ghost cells
for (int i = 1; i <= N; ++i)
    u[i] = rank * N + i - 1;                // a linear function: the global index

int left  = rank > 0        ? rank - 1 : MPI_PROC_NULL;
int right = rank < size - 1 ? rank + 1 : MPI_PROC_NULL;

// bogus ghost values - except at the global boundary (linear extrapolation)
u[0]     = left  == MPI_PROC_NULL ? -1.0     : -1000.0;
u[N + 1] = right == MPI_PROC_NULL ? size * N : -1000.0;

// 1. copy to send buffers: not needed, the boundary values are single numbers
// 2. start all communication - tag 0: data moving to the right, tag 1: data moving to the left
MPI_Request requests[4];
MPI_Irecv(&u[0],     1, MPI_DOUBLE, left,  0, MPI_COMM_WORLD, &requests[0]);
MPI_Irecv(&u[N + 1], 1, MPI_DOUBLE, right, 1, MPI_COMM_WORLD, &requests[1]);
MPI_Isend(&u[1],     1, MPI_DOUBLE, left,  1, MPI_COMM_WORLD, &requests[2]);
MPI_Isend(&u[N],     1, MPI_DOUBLE, right, 0, MPI_COMM_WORLD, &requests[3]);

// 3. bulk update: only reads u[1 .. N] - reading a send buffer is allowed
for (int i = 2; i <= N - 1; ++i)
    unew[i] = (u[i - 1] + u[i] + u[i + 1]) / 3.0;

// 4. wait for all communication to complete
MPI_Waitall(4, requests, MPI_STATUSES_IGNORE);

// 5. copy into ghost cells: not needed, they were received in place
// 6. update the cells next to the ghost cells
unew[1] = (u[0]     + u[1] + u[2])     / 3.0;
unew[N] = (u[N - 1] + u[N] + u[N + 1]) / 3.0;

double maxdev = 0.0;
for (int i = 1; i <= N; ++i) {
    double dev = unew[i] - u[i];
    if (dev < 0.0) dev = -dev;
    if (dev > maxdev) maxdev = dev;
}
printf("rank %d: ghost cells %5.1f %5.1f, max. deviation %g\n", rank, u[0], u[N + 1], maxdev);

> 🤔 **Predict:** What happens if you move the `MPI_Waitall()` behind step 6? Try it.

Note that the send buffers `u[1]` and `u[N]` are *read* during the bulk update while the sends are still pending.
This is allowed since MPI 3.0 — modifying them is not.

As in the blocking version, each direction has its own tag.
With periodic boundaries and only two processes, the left and the right neighbor are the same rank — then only the tag tells the two messages apart.

### Waiting for Some of the Requests *(optional)*

When the pending operations can be processed independently, it is often better to handle each one as soon as it has completed:

```c
int MPI_Waitany(int count, MPI_Request requests[], int *index, MPI_Status *status);
int MPI_Testany(int count, MPI_Request requests[], int *index, int *flag, MPI_Status *status);

int MPI_Waitsome(int incount, MPI_Request requests[], int *outcount, int indices[], MPI_Status statuses[]);
int MPI_Testsome(int incount, MPI_Request requests[], int *outcount, int indices[], MPI_Status statuses[]);
```

* `MPI_Waitany()` waits until at least one of the requests has completed, completes **exactly one** of them and returns its position in `index`; `MPI_Testany()` checks whether one has completed.
* `MPI_Waitsome()` waits until **at least one** of the requests has completed and returns all completed ones in `indices[0 … outcount-1]`; `MPI_Testsome()` returns immediately, possibly with `outcount = 0`.
* Completed requests are set to `MPI_REQUEST_NULL` and are ignored by subsequent calls, so the functions can be called in a loop over the same array.
  Once all requests are `MPI_REQUEST_NULL`, the index is set to `MPI_UNDEFINED`.
* If several requests have completed, `MPI_Waitany()` may pick any of them — a manager that re-posts the receive for the chosen worker every time can starve the others.
  `MPI_Waitsome()` returns all completed requests at once; handling all of them before waiting again serves every worker.

A typical example is a manager that collects results from workers in the order in which they become ready.
Here, the higher the rank, the earlier its result is ready.

> 🤔 **Predict:** In which order does rank 0 print the results?

In [ ]:
%%mpi -n 4 -o code/p2p-waitany.c
if (rank != 0) {
    double t0 = MPI_Wtime();
    while (MPI_Wtime() - t0 < 0.2 * (size - rank))
        ;                                       // "compute" the result
    int result = rank * rank;
    MPI_Send(&result, 1, MPI_INT, 0, 0, MPI_COMM_WORLD);
} else {
    int *results = malloc(size * sizeof(int));
    MPI_Request *requests = malloc(size * sizeof(MPI_Request));

    requests[0] = MPI_REQUEST_NULL;             // nothing to receive from myself
    for (int source = 1; source < size; ++source)
        MPI_Irecv(&results[source], 1, MPI_INT, source, 0, MPI_COMM_WORLD, &requests[source]);

    for (int i = 1; i < size; ++i) {
        int index;
        MPI_Waitany(size, requests, &index, MPI_STATUS_IGNORE);
        printf("rank 0: result %d from rank %d\n", results[index], index);
    }

    free(results);
    free(requests);
}

Rank 0 prints the results in the order in which they arrive — from rank 3, 2, 1 — not in the order in which the receives were posted.

### Nonblocking `MPI_Sendrecv()` *(optional)*

MPI 4.0 added nonblocking versions of the combined calls:

```c
int MPI_Isendrecv(const void *sendbuf, int sendcount, MPI_Datatype sendtype, int dest,   int sendtag,
                  void       *recvbuf, int recvcount, MPI_Datatype recvtype, int source, int recvtag,
                  MPI_Comm comm, MPI_Request *request);
int MPI_Isendrecv_replace(void *buf, int count, MPI_Datatype datatype,
                          int dest, int sendtag, int source, int recvtag,
                          MPI_Comm comm, MPI_Request *request);
```

They are only available in recent MPI implementations.
The cell checks the version of the standard the library claims to implement (`MPI_VERSION`) — some libraries already provide the new calls while still reporting an older version.
Open MPI 5, for example, reports MPI 3.1 but implements `MPI_Isendrecv()`, so the cell checks for it explicitly:

In [ ]:
%%mpi -n 4 -o code/p2p-isendrecv.c
#if MPI_VERSION >= 4 || (defined(OMPI_MAJOR_VERSION) && OMPI_MAJOR_VERSION >= 5)
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;
int sendval = rank, recvval = -1;
MPI_Request request;

MPI_Isendrecv(&sendval, 1, MPI_INT, right, 0,
              &recvval, 1, MPI_INT, left,  0, MPI_COMM_WORLD, &request);
// work that does not touch sendval or recvval could be done here
MPI_Wait(&request, MPI_STATUS_IGNORE);

printf("rank %d: received %d\n", rank, recvval);
#else
if (rank == 0)
    printf("MPI_Isendrecv() requires MPI 4.0, this library implements MPI %d.%d\n", MPI_VERSION, MPI_SUBVERSION);
#endif

### Persistent Requests *(optional)*

Iterative codes often exchange the same messages — same buffers, partners and tags — in every iteration.
A **persistent request** sets up such an operation once and starts it as often as needed:

```c
int MPI_Send_init(const void *buf, int count, MPI_Datatype datatype, int dest,   int tag,
                  MPI_Comm comm, MPI_Request *request);
int MPI_Recv_init(void *buf,       int count, MPI_Datatype datatype, int source, int tag,
                  MPI_Comm comm, MPI_Request *request);
int MPI_Start(MPI_Request *request);
int MPI_Startall(int count, MPI_Request requests[]);
int MPI_Request_free(MPI_Request *request);
```

* `MPI_Send_init()` and `MPI_Recv_init()` take the arguments of `MPI_Isend()` and `MPI_Irecv()`, but don't communicate yet — the new request is *inactive*.
* `MPI_Start()` and `MPI_Startall()` start the operations, which are completed with the usual `MPI_Wait*()` and `MPI_Test*()` calls.
* After completion, the request is **not** set to `MPI_REQUEST_NULL` but becomes inactive again, and can be started once more.
* When it is no longer needed, the request is freed with `MPI_Request_free()`.
* There are persistent variants of all send modes (`MPI_Bsend_init()`, `MPI_Ssend_init()`, `MPI_Rsend_init()`), and persistent operations match any send or receive on the other side.

The buffers are fixed when the request is created, but their contents may change between the iterations.
MPI can do the setup work once and may optimize the repeated transfer — the persistent collectives in later notebooks build on the same idea.

> 🤔 **Predict:** What does each rank receive in the last iteration?

In [ ]:
%%mpi -n 4 -o code/p2p-persistent.c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;
int sendval = 0, recvval = -1;

// set up the ring shift once ...
MPI_Request requests[2];
MPI_Recv_init(&recvval, 1, MPI_INT, left,  0, MPI_COMM_WORLD, &requests[0]);
MPI_Send_init(&sendval, 1, MPI_INT, right, 0, MPI_COMM_WORLD, &requests[1]);

// ... and start it in every iteration
for (int iter = 0; iter < 3; ++iter) {
    sendval = 100 * iter + rank;            // the buffer is fixed, its contents may change
    MPI_Startall(2, requests);
    MPI_Waitall(2, requests, MPI_STATUSES_IGNORE);
}

printf("rank %d: received %d in the last iteration\n", rank, recvval);

MPI_Request_free(&requests[0]);
MPI_Request_free(&requests[1]);

Every rank receives `200 + left`, the value its left neighbor sent in the last iteration — the same two requests were used for all three iterations.

## Summary: Nonblocking Point-to-Point Communication

* `MPI_Isend()` / `MPI_Irecv()` start a communication and return a **request**; the return does not imply completion.
* Complete the request with `MPI_Wait*()` (blocks) or `MPI_Test*()` (returns immediately): `…`, `…all`, `…any`, `…some`.
* Until completion, a send buffer must not be modified and a receive buffer must not be used at all.
* **All requests must be completed** — a request that is never waited for or tested to completion is a bug, even if the program seems to work.
* Typical bugs:
    * a buffer goes out of scope while its operation is still pending — e.g., a local variable of a function that returns before the wait,
    * a request variable is overwritten by a new operation before the old one has completed — the old request is lost,
    * a send buffer is modified before the send has completed — the receiver may get the old values, the new ones, or a mix of both.
* Potentials:
    * avoiding deadlocks, synchronization and idle times,
    * overlapping multiple communication operations, e.g., sends and receives with all neighbors at once,
    * overlapping communication with work — possible, but **not guaranteed** by the MPI standard.

## Exercises

### Exercise 1: Ring Sum

Compute the sum of all ranks with a ring:
every rank starts with its own rank as `value`, and in each step passes `value` to its right neighbor and receives a new one from its left neighbor with `MPI_Sendrecv_replace()`.
Every rank adds up all values it sees.

> 🤔 **Predict:** How many steps are needed until every rank knows the sum? Which rank knows it first?

<details>
<summary>Solution</summary>

```c
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;
int value = rank, sum = rank;

for (int step = 1; step < size; ++step) {
    MPI_Sendrecv_replace(&value, 1, MPI_INT, right, 0, left, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    sum += value;
}

printf("rank %d: sum = %d, expected %d\n", rank, sum, size * (size - 1) / 2);
```

After `size - 1` steps, all ranks know the sum at the same time.
This is what the collective operation `MPI_Allreduce()` does in a single call — usually with a much faster algorithm.
</details>

In [ ]:
%%mpi -n 4
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;
int value = rank, sum = rank;

// TODO: pass value around the ring and add up all values

printf("rank %d: sum = %d, expected %d\n", rank, sum, size * (size - 1) / 2);

### Exercise 2: Ordered Calls vs. `MPI_Sendrecv()`

In Exercise 5 of the previous notebook, you made the ring shift deadlock-free by ordering the calls: even ranks send first, odd ranks receive first.
Compare this with `MPI_Sendrecv()`:

* implement the ring shift in both ways — variant 0 with `MPI_Send()` and `MPI_Recv()` in even/odd order, variant 1 with a single `MPI_Sendrecv()`,
* the cell measures the time per exchange for messages from 1 KiB to 4 MiB — run it with 2, 3 and 4 processes,
* check that both variants are *safe*: run the cell once more with `-f DMPI_Send=MPI_Ssend`.

> 🤔 **Predict:** Which variant is faster — for small messages, for messages above the eager limit, for large ones? What happens with `-n 1`?

<details>
<summary>Solution</summary>

Replace the `TODO` in the loop over `r` by

```c
if (variant == 0) {
    if (rank % 2 == 0) {
        MPI_Send(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD);
        MPI_Recv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    } else {
        MPI_Recv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD);
    }
} else {
    MPI_Sendrecv(sendbuf, n, MPI_INT, right, 0,
                 recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
}
```

For example, with `-n 2` on an Apple M2 Pro with Open MPI (shared memory, eager limit 4 KiB):

```
     bytes   even/odd [us]   MPI_Sendrecv [us]
      1024            0.46                0.35
     16384            2.88                1.83
    262144           18.18               17.82
   4194304          360.44              309.88
```

`MPI_Sendrecv()` is the simplest solution and never slower:

* **Small messages:** below the eager limit, the sends of the even ranks return immediately, so the two phases of the even/odd order overlap anyway — the difference is small.
* **Messages above the eager limit** (here 16 KiB): the even/odd order really needs two phases, first from the even to the odd ranks, then the other way round.
  With `MPI_Sendrecv()`, every rank sends and receives at the same time — up to twice as fast.
  With an odd number of processes, the even/odd order even needs three phases (Exercise 5 of the previous notebook), `MPI_Sendrecv()` still one.
* **Large messages:** on a single node, the transfers are limited by the memory bandwidth, which concurrent copies have to share, and the gain shrinks.
  Across a network, whose links carry data in both directions at the same time, the gain can be larger.

With `-f DMPI_Send=MPI_Ssend`, both variants still complete — they are safe.
The even/odd times for small messages grow, since a synchronous send waits for the receiver even when the message could be sent eagerly; `MPI_Sendrecv()` is not affected by the macro.

With `-n 1`, the even/odd variant deadlocks: rank 0 sends to itself before it receives, which hangs as soon as the message exceeds the eager limit.
`MPI_Sendrecv()` works — a rank can exchange data with itself.
</details>


In [ ]:
%%mpi -n 4 -T 30
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

if (rank == 0)
    printf("%10s %15s %19s\n", "bytes", "even/odd [us]", "MPI_Sendrecv [us]");

for (int n = 1 << 8; n <= (1 << 20); n *= 16) {
    int *sendbuf = malloc(n * sizeof(int));
    int *recvbuf = malloc(n * sizeof(int));
    const int reps = n <= (1 << 12) ? 1000 : 50;
    double time[2];

    for (int i = 0; i < n; ++i)
        sendbuf[i] = rank;

    for (int variant = 0; variant < 2; ++variant) {
        double start = 0.0;
        recvbuf[0] = -1;

        // the first reps / 10 exchanges are a warm-up and not timed
        for (int r = -reps / 10; r < reps; ++r) {
            if (r == 0) {
                MPI_Barrier(MPI_COMM_WORLD);    // all ranks start together
                start = MPI_Wtime();
            }

            // TODO: send sendbuf to the right and receive recvbuf from the left
            //       variant 0: even ranks send first, odd ranks receive first
            //       variant 1: a single MPI_Sendrecv()
        }
        MPI_Barrier(MPI_COMM_WORLD);            // the exchange is done once the last rank is done
        time[variant] = (MPI_Wtime() - start) / reps;

        if (recvbuf[0] != left)
            printf("rank %d: variant %d did not receive the data of rank %d\n", rank, variant, left);
    }

    if (rank == 0)
        printf("%10zu %15.2f %19.2f\n", n * sizeof(int), 1e6 * time[0], 1e6 * time[1]);

    free(sendbuf);
    free(recvbuf);
}

### Exercise 3: MVM with Overlapping Communication

In the MVM case study, the ring shift and the computation take turns.
Overlap them using **double buffering**: while the current chunk `x` is used for the computation, the next chunk is already received into a second buffer `next`.

* Before the computation, start an `MPI_Irecv()` of the next chunk into `next` and an `MPI_Isend()` of `x` (except in the last step).
* After the computation, wait for both requests and swap the pointers `x` and `next`.
* Compare the time of the MVM with the original version — with `-n 4` and also with fewer and more processes. Run each version a few times, since single measurements fluctuate.

<details>
<summary>Solution</summary>

```c
for (int step = 0; step < size; ++step) {
    const int more = step < size - 1;       // is there a next chunk?
    MPI_Request requests[2];

    // start the ring shift: x may be read, but not modified while it is being sent
    if (more) {
        MPI_Irecv(next, nmax, MPI_DOUBLE, lower, 0, MPI_COMM_WORLD, &requests[0]);
        MPI_Isend(x,    nmax, MPI_DOUBLE, upper, 0, MPI_COMM_WORLD, &requests[1]);
    }

    const int cs = CHUNK_START(owner), nc = CHUNK_SIZE(owner);
    for (int r = 0; r < nlocal; ++r)
        for (int c = 0; c < nc; ++c)
            y[r] += a[(size_t)r * N + cs + c] * x[c];

    if (more) {
        MPI_Waitall(2, requests, MPI_STATUSES_IGNORE);
        double *tmp = x;  x = next;  next = tmp;
        owner = (owner + 1) % size;
    }
}
```

With only a few small chunks, the gain is small: the communication of about `N` doubles per rank is cheap compared to the `N² / size` multiply-adds — and whether MPI progresses the transfer in the background is up to the implementation.
On a single node, the difference is usually within the measurement noise: the cores copy the data themselves, so nothing moves it while they compute.
And once there are more processes than (performance) cores, the time is dominated by waiting for the slowest rank in every step, which no overlap can hide.
The overlap can pay off across a network, where the network interface transfers the data, once the communication takes a noticeable share of the runtime.
</details>

In [ ]:
%%mpi -n 4
const int N = 2000;

const int num = N / size, rest = N % size;
#define CHUNK_SIZE(r)  (num + ((r) < rest ? 1 : 0))
#define CHUNK_START(r) ((r) * num + ((r) < rest ? (r) : rest))

const int nlocal = CHUNK_SIZE(rank);
const int start  = CHUNK_START(rank);
const int nmax   = CHUNK_SIZE(0);

double *a    = malloc((size_t)nlocal * N * sizeof(double));
double *y    = calloc(nlocal, sizeof(double));
double *x    = calloc(nmax, sizeof(double));    // the chunk used for the computation
double *next = calloc(nmax, sizeof(double));    // the chunk being received

for (int r = 0; r < nlocal; ++r)
    for (int c = 0; c < N; ++c)
        a[(size_t)r * N + c] = 1.0 / (start + r + c + 1);
for (int c = 0; c < nlocal; ++c)
    x[c] = start + c + 1;

int upper = (rank - 1 + size) % size;
int lower = (rank + 1) % size;
int owner = rank;

MPI_Barrier(MPI_COMM_WORLD);
double t0 = MPI_Wtime();

for (int step = 0; step < size; ++step) {
    // TODO: start receiving the next chunk into next and sending x (not in the last step)

    const int cs = CHUNK_START(owner), nc = CHUNK_SIZE(owner);
    for (int r = 0; r < nlocal; ++r)
        for (int c = 0; c < nc; ++c)
            y[r] += a[(size_t)r * N + cs + c] * x[c];

    // TODO: complete the communication, swap x and next, update owner
}

double t1 = MPI_Wtime();

double maxerr = 0.0;
for (int r = 0; r < nlocal; ++r) {
    double ref = 0.0;
    for (int c = 0; c < N; ++c)
        ref += 1.0 / (start + r + c + 1) * (c + 1);
    double err = (y[r] - ref) / ref;
    if (err < 0.0) err = -err;
    if (err > maxerr) maxerr = err;
}
printf("rank %d: rows [%4d, %4d), max. relative error %.1e, MVM %.3f ms\n",
       rank, start, start + nlocal, maxerr, 1e3 * (t1 - t0));

free(a);
free(y);
free(x);
free(next);

### Exercise 4: Does Your MPI Overlap Communication and Computation?

Find out whether your MPI library transfers a large message in the background, while the sender is busy with computation that does not call MPI.

* Rank 0 starts an `MPI_Isend()` of 64 MiB to rank 1, works for `ms` milliseconds (a busy loop with `MPI_Wtime()`), and then calls `MPI_Wait()`.
  It measures the time spent in `MPI_Wait()`.
* Rank 1 posts the matching receive immediately and afterwards sends a small acknowledgment back, so that both processes start the next round together.
* Repeat this for `ms = 0, 20, 40, …, 100`. The first round may include one-time setup costs — run the loop twice if its first line looks odd.

If the transfer progresses in the background, the time in `MPI_Wait()` drops to almost zero once the work takes longer than the transfer.
If it stays constant, the message is only transferred while rank 0 is inside an MPI call.
In that case, try calling `MPI_Test()` from time to time inside the work loop.

<details>
<summary>Solution</summary>

```c
const int n = 1 << 26;      // 64 MiB
char *buf = calloc(n, 1);

if (rank == 0)
    printf("%10s %10s %10s\n", "work [ms]", "total [ms]", "wait [ms]");

for (int ms = 0; ms <= 100; ms += 20) {
    if (rank == 0) {
        MPI_Request request;
        int ack;

        double t0 = MPI_Wtime();
        MPI_Isend(buf, n, MPI_BYTE, 1, 0, MPI_COMM_WORLD, &request);
        while (MPI_Wtime() - t0 < 1e-3 * ms)
            ;                               // work without any MPI calls
        double t1 = MPI_Wtime();
        MPI_Wait(&request, MPI_STATUS_IGNORE);
        double t2 = MPI_Wtime();

        MPI_Recv(&ack, 1, MPI_INT, 1, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("%10d %10.1f %10.1f\n", ms, 1e3 * (t2 - t0), 1e3 * (t2 - t1));
    } else if (rank == 1) {
        int ack = 0;
        MPI_Recv(buf, n, MPI_BYTE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(&ack, 1, MPI_INT, 0, 1, MPI_COMM_WORLD);
    }
}

free(buf);
```

The result depends on the MPI library, its configuration and the transport.
Many implementations only make progress inside MPI calls; some can be configured to use a progress thread or offload the transfer to the network hardware.
On a single node, the shared-memory transport often lets the *receiver* copy the data directly from the sender's buffer (single-copy, e.g. via CMA or XPMEM).
The transfer then progresses while the sender computes, and the time in `MPI_Wait()` drops to a small remainder — e.g. from 14 ms to about 2.5 ms with Open MPI on a laptop, since the final handshake still needs the sender inside an MPI call.
The same library may behave quite differently across a network.
</details>

In [ ]:
%%mpi -n 2
const int n = 1 << 26;      // 64 MiB
char *buf = calloc(n, 1);

// TODO: for ms = 0, 20, ..., 100: rank 0 sends buf to rank 1 with MPI_Isend(),
//       works for ms milliseconds, waits for the send to complete,
//       and prints the time spent in MPI_Wait()

free(buf);

## Summary

The point-to-point calls of this notebook extend the beginner's toolbox:

| Call | Purpose |
|---|---|
| `MPI_Ssend()`, `MPI_Bsend()` | send in synchronous or buffered mode |
| `MPI_Buffer_attach()`, `MPI_Buffer_detach()` | provide a buffer for buffered sends |
| `MPI_Sendrecv()` | send and receive at the same time, without deadlocks |
| `MPI_Sendrecv_replace()` | the same, with a single buffer |
| `MPI_PROC_NULL` | a neighbor that does not exist |
| `MPI_Probe()`, `MPI_Iprobe()` | check for a message without receiving it |
| `MPI_Isend()`, `MPI_Irecv()` | start a send or receive and return immediately |
| `MPI_Wait()`, `MPI_Test()` | complete one request |
| `MPI_Waitall()`, `MPI_Waitany()`, `MPI_Waitsome()` (and `MPI_Test…`) | complete several requests |
| `MPI_Send_init()`, `MPI_Recv_init()`, `MPI_Start()` | set up a communication once, start it repeatedly |

* A blocking call returns once its buffer can be reused; a nonblocking call returns immediately, and its buffer must be left alone until the request has completed.
* Whether `MPI_Send()` waits for the receiver is up to the implementation — write **safe** programs that work even if it does.
* Neighbor exchanges — ring shifts, ghost cell exchange — are done with `MPI_Sendrecv()` or nonblocking communication, and `MPI_PROC_NULL` at open boundaries.
* Nonblocking communication allows to overlap communication with computation, but the MPI standard does not guarantee that the transfer actually progresses in the background.

## Outlook

* Collecting and distributing data among *all* processes, like the ring sum of Exercise 1 → **collective communication** (`MPI_Bcast()`, `MPI_Reduce()`, `MPI_Allreduce()`, …).
* Sending non-contiguous data, like the column boundary layers of a 2D tile, without copying it by hand → **derived data types**.
* Finding the neighbors in a 2D or 3D domain decomposition, including `MPI_PROC_NULL` at the boundaries → **Cartesian topologies** (`MPI_Cart_create()`, `MPI_Cart_shift()`).

[Back to the overview](../overview.ipynb)